# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarBabar02/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: The Freshness Multiplier

The paper reports a large increase in impressions for refreshed mature pages and describes freshness as one of the strongest observed signals in the dataset.

**Methodology question:** Where does the outcome label come from? I would want to confirm exactly how a page is classified as “refreshed” and how the later impression outcome is measured. I would also ask whether the comparison controls for page age, position, and other differences between refreshed and non-refreshed pages.

**Validation question:** Does the validation design support the strength of the claim? The paper describes this as a pattern study and notes that the results do not prove cause and effect. Therefore, the finding is useful as an observed association, but I would not interpret the measured impression increase as proof that refreshing itself caused the increase.

### Finding 2: The CTR Cliff

The paper reports that CTR changes substantially across search-position tiers and uses position as an important part of its analysis.

**Methodology question:** Where does the label or outcome come from? I would want to confirm that CTR is calculated from observed Search Console clicks and impressions for the same measurement window, and that pages with very different impression volumes are handled consistently.

**Validation question:** Does the validation design carry the claim? Position and CTR are naturally related, so I would want to check whether the comparison controls for differences in page mix and search visibility. The paper's methodology says its headline findings are based mainly on direct aggregate comparisons, so the result should be read as an observed portfolio pattern rather than proof that position alone causes the CTR change.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# Section 2 — Honest time-aware validation
# Recreate the Week-5 modeling dataset

import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

# Load Hugging Face data
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

rel = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

query = f"""
WITH base AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions,
        ga4_total_engagement_sec,
        sessions_organic,
        sessions_direct,
        sessions_referral,
        sessions_social,
        sessions_paid,
        sessions_ai,
        scroll_events
    FROM {rel}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-30'
),

current_data AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions,
        ga4_total_engagement_sec,
        sessions_organic,
        sessions_direct,
        sessions_referral,
        sessions_social,
        sessions_paid,
        sessions_ai,
        scroll_events,

        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS ctr

    FROM base
),

next_day AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS next_ctr

    FROM base
)

SELECT
    c.*,
    n.next_ctr,

    CASE
        WHEN c.ctr IS NOT NULL
             AND n.next_ctr IS NOT NULL
             AND n.next_ctr > c.ctr
        THEN 1
        ELSE 0
    END AS target

FROM current_data c

LEFT JOIN next_day n
    ON c.client_hash_id = n.client_hash_id
    AND c.content_hash_id = n.content_hash_id
    AND n.report_date = c.report_date + INTERVAL 1 DAY

WHERE c.gsc_impressions > 0
  AND c.gsc_avg_position > 0
  AND n.next_ctr IS NOT NULL
  AND c.gsc_impressions >= 5
"""

print("Running DuckDB query...")

training_df = con.sql(query).df()

print("Original dataset shape:", training_df.shape)


# Same Week-5 sampling
MAX_ROWS = 300000

if len(training_df) > MAX_ROWS:
    training_df = (
        training_df
        .sample(
            n=MAX_ROWS,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )

print("Final ML dataset shape:", training_df.shape)


# Same Week-5 features
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]


# Make sure dates are sorted
training_df = training_df.sort_values(
    "report_date"
).reset_index(drop=True)

X = training_df[feature_cols].copy()
y = training_df["target"].astype(int)


# Time-aware split
# First 80% of dates = training
# Last 20% of dates = testing

unique_dates = sorted(
    training_df["report_date"].unique()
)

split_index = int(len(unique_dates) * 0.80)

train_dates = unique_dates[:split_index]
test_dates = unique_dates[split_index:]

train_mask = training_df["report_date"].isin(train_dates)
test_mask = training_df["report_date"].isin(test_dates)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()


print("\n--- Honest Time-Aware Split ---")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training dates:", train_dates[0], "to", train_dates[-1])
print("Test dates:", test_dates[0], "to", test_dates[-1])


# Logistic Regression
honest_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])


honest_model.fit(
    X_train,
    y_train
)

honest_probability = (
    honest_model.predict_proba(X_test)[:, 1]
)

honest_roc_auc = roc_auc_score(
    y_test,
    honest_probability
)


print("\n--- Honest Validation Result ---")
print(
    "Time-aware ROC AUC:",
    round(honest_roc_auc, 4)
)

Running DuckDB query...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Original dataset shape: (2409018, 20)
Final ML dataset shape: (300000, 20)

--- Honest Time-Aware Split ---
Training rows: 233181
Test rows: 66819
Training dates: 2026-03-01 00:00:00 to 2026-03-23 00:00:00
Test dates: 2026-03-24 00:00:00 to 2026-03-29 00:00:00

--- Honest Validation Result ---
Time-aware ROC AUC: 0.7742


### Validation comparison

The Week-5 Logistic Regression model had a ROC AUC of **0.7471** on its held-out client-grouped test set.

For this audit, I re-ran the same modeling approach using a **time-aware split**, where earlier dates were used for training and later dates were used for testing.

The time-aware validation produced a ROC AUC of **0.7742**.

| Evaluation | ROC AUC |
|---|---:|
| Week-5 client-grouped validation | 0.7471 |
| W06 time-aware validation | 0.7742 |

The time-aware result is higher than the Week-5 result. This is an observed difference between two validation designs and does not prove that the model will perform the same way on future data.

The W06 test period covers **March 24 to March 29, 2026**, while training uses **March 1 to March 23, 2026**. This keeps later dates separate from earlier training data.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# Leakage audit for the final Week-5 feature set

leakage_keywords = [
    "target",
    "next_",
    "future",
    "trend",
    "label",
    "outcome"
]

leakage_check = []

for feature in feature_cols:
    matched_terms = [
        word for word in leakage_keywords
        if word.lower() in feature.lower()
    ]

    leakage_check.append({
        "feature": feature,
        "possible_leakage_keyword": (
            ", ".join(matched_terms)
            if matched_terms else "None"
        )
    })

leakage_audit = pd.DataFrame(leakage_check)

display(leakage_audit)

print("\nExcluded future/label-derived columns:")
print([
    col for col in training_df.columns
    if any(word in col.lower() for word in leakage_keywords)
])

,feature,possible_leakage_keyword
0,gsc_impressions,None
1,gsc_clicks,None
2,ctr,None
3,gsc_avg_position,None
4,ga4_pageviews,None
5,ga4_sessions,None
6,ga4_engaged_sessions,None
7,ga4_total_engagement_sec,None
8,sessions_organic,None
9,sessions_direct,None



Excluded future/label-derived columns:
['next_ctr', 'target']


### Leakage audit

The final model features were checked for future-looking and label-derived fields.

The target and `next_ctr` are excluded from the feature set. Trend-related fields were also not used as model features. The model uses observed search-performance and engagement signals such as impressions, clicks, CTR, average position, sessions, and scroll events.

The client identifier is not included as a model feature.

This audit supports the conclusion that the final feature set does not intentionally include the future target. However, the features are still observational measurements, so their relationship with the target should be interpreted as directional rather than causal.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [8]:
# Real failure examples from the honest time-aware test set

error_examples = training_df.loc[test_mask][
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "next_ctr",
        "target"
    ]
].copy()

error_examples["model_probability"] = honest_probability

error_examples["prediction"] = (
    error_examples["model_probability"] >= 0.5
).astype(int)

error_examples["error_type"] = np.select(
    [
        (error_examples["target"] == 1) &
        (error_examples["prediction"] == 0),

        (error_examples["target"] == 0) &
        (error_examples["prediction"] == 1)
    ],
    [
        "False Negative",
        "False Positive"
    ],
    default="Correct"
)

print("Error counts:")

display(
    error_examples["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

print("\nFalse Positive examples:")

display(
    error_examples[
        error_examples["error_type"] == "False Positive"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
    .head(5)
)

print("\nFalse Negative examples:")

display(
    error_examples[
        error_examples["error_type"] == "False Negative"
    ]
    .sort_values(
        "model_probability",
        ascending=True
    )
    .head(5)
)

Error counts:


,error_type,count
0,Correct,52033
1,False Positive,12083
2,False Negative,2703



False Positive examples:


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,next_ctr,target,model_probability,prediction,error_type
296897,2026-03-29,client_23a62021009f63c4,content_e943d753806d7af3,8327,21,0.002522,8.844001,0.002395,0,1.0,1,False Positive
294111,2026-03-29,client_e547b89c05043229,content_0e03de7680314cd5,25582,49,0.001915,2.390978,0.000511,0,1.0,1,False Positive
247074,2026-03-25,client_e547b89c05043229,content_0e03de7680314cd5,10025,30,0.002993,2.652269,0.002787,0,1.0,1,False Positive
238792,2026-03-24,client_e547b89c05043229,content_8d7d99f109e19aa2,20722,7,0.000338,2.434514,0.000295,0,1.0,1,False Positive
262441,2026-03-26,client_e547b89c05043229,content_ec2e0346994fb5a5,6905,41,0.005938,2.972918,0.005809,0,1.0,1,False Positive



False Negative examples:


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,next_ctr,target,model_probability,prediction,error_type
254124,2026-03-25,client_20259bd6705d81d4,content_97a4b09ecfd9cacf,58,0,0.0,128.120690,0.045455,1,0.010848,0,False Negative
254691,2026-03-25,client_20259bd6705d81d4,content_efcff2c452122c22,30,0,0.0,108.400000,0.032258,1,0.019247,0,False Negative
244477,2026-03-25,client_23a62021009f63c4,content_75197363ba40652f,106,0,0.0,115.433962,0.017857,1,0.021103,0,False Negative
252663,2026-03-25,client_20259bd6705d81d4,content_352b7f4471e11f22,295,0,0.0,123.267797,0.009259,1,0.044387,0,False Negative
253146,2026-03-25,client_f623b01661d4bfe4,content_969352f4e556bad4,11,0,0.0,77.090909,0.034483,1,0.052606,0,False Negative


## 4. Failure examples and claim rewrite
**Earlier claim:**

“The Logistic Regression model significantly outperforms the Week-4 baseline and is a better way to identify CTR improvement opportunities.”

**Safer claim:**

“On the held-out client-grouped test sample, the Logistic Regression model showed higher measured Precision@20 and Precision@50 than the Week-4 rule-based baseline. This is an observed result on this evaluation sample and suggests that the model may provide useful decision-support for prioritizing content review. It does not prove that the model will outperform the baseline for every future client or that the model causes CTR improvement.”


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.